In [7]:
import sys
import sentence_transformers
import sentence_transformers.models

# ==============================================================================
# O TRUQUE DE MESTRE 2.0: Mapeando todas as sub-pastas antigas
# ==============================================================================
sys.modules['sentence_transformers.sentence_transformer'] = sentence_transformers
sys.modules['sentence_transformers.sentence_transformer.model'] = sentence_transformers.models

from meu_bertopic import BERTopic 

caminho_modelo = "modelagem_final/bertopic_model"

print("Carregando o BERTopic...")
modelo = BERTopic.load(caminho_modelo)
print("✅ Modelo carregado com sucesso!")

Carregando o BERTopic...


AttributeError: Can't get attribute 'SentenceTransformer' on <module 'sentence_transformers.models' from '/home/ester/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sentence_transformers/models/__init__.py'>

In [1]:
import pandas as pd
import random
from meu_bertopic import BERTopic

/home/ester/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
caminho_modelo = "modelagem_final/bertopic_model"
arquivo_saida = "relatorio_modelagem.txt"
modelo = BERTopic.load(caminho_modelo)

ModuleNotFoundError: No module named 'sentence_transformers.sentence_transformer'

In [ ]:
data_frame = pd.read_csv("../data/df_clean_text.csv")
df = data_frame[data_frame['clean_text'].notna()].reset_index(drop=True)
df.drop_duplicates(subset=['clean_text'], inplace=True)
docs = df['clean_text'].tolist()

In [ ]:
topicos_previstos, _ = modelo.transform(docs)
info_topicos = modelo.get_topic_info()
topicos_validos = [t for t in info_topicos['Topic'].tolist() if t != -1]

In [ ]:
with open(arquivo_saida, "w", encoding="utf-8") as f:
    f.write(f"{'='*80}\n")
    f.write(f" RELATÓRIO QUALITATIVO DA MODELAGEM\n")
    f.write(f" Total de Tópicos Encontrados: {len(topicos_validos)}\n")
    f.write(f"{'='*80}\n\n")

    for topico in topicos_validos:
        
        # 1. Extrações do Tópico
        palavras_com_pesos = modelo.get_topic(topico)
        top_10_palavras = [palavra for palavra, peso in palavras_com_pesos][:10]
        docs_representativos = modelo.get_representative_docs(topico)

        # 2. Filtra todos os docs deste tópico para o sorteio
        docs_do_topico = [doc for doc, t in zip(docs, topicos_previstos) if t == topico]
        # Remove os representativos para não duplicar no sorteio
        docs_disponiveis = [doc for doc in docs_do_topico if doc not in docs_representativos]

        # Sorteia 7 (ou menos, se o tópico for muito pequeno)
        qtd_sorteio = min(7, len(docs_disponiveis))
        docs_aleatorios = random.sample(docs_disponiveis, qtd_sorteio)

        # 3. Escrita no arquivo TXT
        f.write(f"{'='*80}\n")
        f.write(f" TÓPICO {topico}\n")
        f.write(f"{'='*80}\n")

        f.write(f"\n🔹 TOP 10 PALAVRAS-CHAVE:\n")
        f.write(" • " + ", ".join(top_10_palavras) + "\n\n")

        f.write(f"🔹 OS 3 DOCUMENTOS MAIS REPRESENTATIVOS (Centro do Cluster):\n")
        for i, doc in enumerate(docs_representativos, start=1):
            f.write(f"  {i}. {doc}\n\n")

        if qtd_sorteio > 0:
            f.write(f"🔹 {qtd_sorteio} DOCUMENTOS ALEATÓRIOS DO MESMO TÓPICO:\n")
            for i, doc in enumerate(docs_aleatorios, start=1):
                f.write(f"  {i}. {doc}\n\n")

        f.write("\n\n") 

print(f"\n✅ Sucesso! O arquivo '{arquivo_saida}' foi criado na sua pasta.")